# Evaluate Fine-tuned DistilBERT NER on the Held-out Test Set

**Inputs:**
- `Explainable_AI_Tweets_Testset.txt` (`id`, `text` - raw tweets, no labels)
- `Explainable_AI_Tweets_Ground_Truth.txt` (`id`, `entities`, `tokenized_text`
  - ground truth, in the original `LABEL/surface text;` format)

**What this notebook does:**
1. Run your fine-tuned model on each raw test tweet, get word-level BIO predictions
2. Reconstruct those BIO tags back into `LABEL/surface text;` strings -
   the same format as the ground truth file
3. Compare predicted entity strings against ground truth, **per tweet**,
   exact match (same label AND identical surface text)
4. Report overall precision/recall/F1, plus a row-by-row diff so you can
   see exactly which entities were missed, invented, or correct

**Important note on text formats:** the ground truth file's text column
is pre-tokenized in a different style than the raw test set (it
space-separates punctuation, and uses `_Mention_`/`_URL_`/`_HASHTAG_`
placeholders instead of `@FakeUsername`/`http://FakeURL`/`#FakeHashtag`).
Because of this, predictions are generated from the **raw test set
text** (matching how the model was trained) and compared against ground
truth as **entity strings**, not via token-position matching - this
sidesteps the tokenization mismatch entirely and compares what actually
matters: did the model find the same entities, written the same way.

## 0. Install dependencies

In [1]:
# !pip install torch transformers nltk pandas -q

## 1. Load your fine-tuned model

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_PATH = "./ner_distilbert_finetuned"  # update if you saved it elsewhere

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()

id2label = model.config.id2label
print("Labels:", list(id2label.values()))

/Users/aniketchoudhary/miniconda3/envs/dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 6796.39it/s]

Labels: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']


## 2. Load the test set and ground truth

`quoting=3` (QUOTE_NONE) avoids pandas misinterpreting stray `"` / `` ` ``
characters inside tweet text as CSV quote characters.

In [5]:
import pandas as pd

TESTSET_PATH = "Explainable AI Tweets Testset.txt"
GROUND_TRUTH_PATH = "Explainable AI Tweets Ground Truth.txt"

test_df = pd.read_csv(TESTSET_PATH, sep="\t", header=None, names=["id", "text"], quoting=3)
gt_df = pd.read_csv(GROUND_TRUTH_PATH, sep="\t", header=None,
                     names=["id", "entities", "gt_text"], quoting=3)
gt_df["entities"] = gt_df["entities"].fillna("")

assert set(test_df["id"]) == set(gt_df["id"]), "Test set and ground truth IDs don't match!"

merged = test_df.merge(gt_df[["id", "entities"]], on="id")
print(f"Loaded {len(merged)} test tweets with ground truth")
merged.head()

Loaded 1450 test tweets with ground truth


,id,text,entities
0,2816,RT @FakeUsername: Christian Bale Thinks The Da...,PER/Christian Bale;MISC/The Dark Knight Rises;
1,2817,RT @FakeUsername: KANYE WEST DARK TWISTED FANT...,PER/KANYE WEST;
2,2818,RT @FakeUsername: #FakeHashtag: Netanyahu told...,PER/Netanyahu;PER/Ackerman;
3,2819,"""``RT @FakeUsername: Netanyahu:'' ``We Need to...",PER/Netanyahu;
4,2820,RT @FakeUsername: RT if you want @FakeUsername...,


## 3. Entity parsing and tokenizer setup

Same `TweetTokenizer` used throughout this project - the model was
trained on its tokenization, so predictions need to use it too.

In [6]:
from nltk.tokenize import TweetTokenizer
import nltk

nltk.download("punkt", quiet=True)
tk = TweetTokenizer()


def parse_entities(entity_str):
    """'PER/louis;ORG/queen;' -> [('PER', 'louis'), ('ORG', 'queen')]"""
    entities = []
    if not entity_str:
        return entities
    for chunk in entity_str.split(";"):
        chunk = chunk.strip()
        if not chunk or "/" not in chunk:
            continue
        label, surface = chunk.split("/", 1)
        entities.append((label.strip().upper(), surface.strip()))
    return entities

## 4. Predict BIO tags for each test tweet

In [7]:
def predict_bio_tags(text):
    """Tokenizes raw text with TweetTokenizer, runs the model, returns
    (tokens, predicted_tags) using first-subword aggregation."""
    tokens = tk.tokenize(text)
    if not tokens:
        return [], []

    enc = tokenizer(tokens, is_split_into_words=True, truncation=True,
                     max_length=64, return_tensors="pt")
    with torch.no_grad():
        logits = model(**enc).logits[0]
    pred_ids = logits.argmax(dim=-1)
    word_ids = enc.word_ids()

    pred_tags = ["O"] * len(tokens)
    seen = set()
    for pos, wid in enumerate(word_ids):
        if wid is not None and wid not in seen:
            seen.add(wid)
            pred_tags[wid] = id2label[pred_ids[pos].item()]
    return tokens, pred_tags

## 5. Reconstruct BIO tags back into entity strings

This is the inverse of the BIO-tagging step used during data prep:
walk the predicted tag sequence, group consecutive `B-X I-X I-X...`
runs into one entity, join the words back with spaces. A fresh `B-`
(even of the same label as the entity right before it) always starts a
new entity, so two adjacent same-type entities aren't merged.

In [8]:
def bio_to_entities(tokens, tags):
    """Returns a list of (label, surface_text) tuples reconstructed
    from a BIO tag sequence."""
    entities = []
    current_label = None
    current_words = []

    def flush():
        if current_label is not None and current_words:
            entities.append((current_label, " ".join(current_words)))

    for tok, tag in zip(tokens, tags):
        if tag == "O":
            flush()
            current_label, current_words = None, []
        elif tag.startswith("B-"):
            flush()
            current_label = tag[2:]
            current_words = [tok]
        elif tag.startswith("I-"):
            label = tag[2:]
            if current_label == label:
                current_words.append(tok)
            else:
                # I- without a preceding matching B- - shouldn't happen
                # from a properly trained model, but handle gracefully
                # rather than dropping it
                flush()
                current_label = label
                current_words = [tok]
    flush()
    return entities


def entities_to_string(entities):
    """[('PER','Christian Bale'), ('MISC','The Dark Knight Rises')]
    -> 'PER/Christian Bale;MISC/The Dark Knight Rises;'"""
    return "".join(f"{label}/{text};" for label, text in entities)

## 6. Run predictions on the full test set

In [10]:
results = []
for _, row in merged.iterrows():
    tokens, pred_tags = predict_bio_tags(row["text"])
    pred_entities = bio_to_entities(tokens, pred_tags)
    pred_string = entities_to_string(pred_entities)
    gt_entities = parse_entities(row["entities"])

    results.append({
        "id": row["id"],
        "text": row["text"],
        "ground_truth_entities": set(gt_entities),
        "predicted_entities": set(pred_entities),
        "ground_truth_string": row["entities"],
        "predicted_string": pred_string,
    })

results_df = pd.DataFrame(results)
results_df.head()

,id,text,ground_truth_entities,predicted_entities,ground_truth_string,predicted_string
0,2816,RT @FakeUsername: Christian Bale Thinks The Da...,"{(PER, Christian Bale), (MISC, The Dark Knight...","{(PER, Christian Bale)}",PER/Christian Bale;MISC/The Dark Knight Rises;,PER/Christian Bale;
1,2817,RT @FakeUsername: KANYE WEST DARK TWISTED FANT...,"{(PER, KANYE WEST)}","{(PER, KANYE)}",PER/KANYE WEST;,PER/KANYE;
2,2818,RT @FakeUsername: #FakeHashtag: Netanyahu told...,"{(PER, Ackerman), (PER, Netanyahu)}","{(PER, Ackerman), (PER, Netanyahu)}",PER/Netanyahu;PER/Ackerman;,PER/Netanyahu;PER/Ackerman;
3,2819,"""``RT @FakeUsername: Netanyahu:'' ``We Need to...","{(PER, Netanyahu)}","{(PER, Netanyahu)}",PER/Netanyahu;,PER/Netanyahu;
4,2820,RT @FakeUsername: RT if you want @FakeUsername...,{},{},,


## 7. Score: exact-match precision / recall / F1

Exact match = same label AND identical surface text. This is computed
directly from the entity sets (not via `seqeval`, since we're comparing
reconstructed strings rather than aligned token sequences - but the
underlying definition of a true/false positive/negative is the same).

In [11]:
tp = fp = fn = 0
for _, row in results_df.iterrows():
    gt = row["ground_truth_entities"]
    pred = row["predicted_entities"]
    tp += len(gt & pred)
    fp += len(pred - gt)
    fn += len(gt - pred)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"True positives:  {tp}")
print(f"False positives: {fp}  (predicted entities not in ground truth)")
print(f"False negatives: {fn}  (ground truth entities the model missed)")
print()
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1:        {f1:.3f}")

True positives:  1032
False positives: 469  (predicted entities not in ground truth)
False negatives: 480  (ground truth entities the model missed)

Precision: 0.688
Recall:    0.683
F1:        0.685


## 8. Per-label breakdown

Same exact-match logic, broken down by entity type, so you can see if
the model is systematically weaker on any one category (e.g. MISC is
typically hardest - it's the least consistent category).

In [12]:
from collections import defaultdict

label_tp = defaultdict(int)
label_fp = defaultdict(int)
label_fn = defaultdict(int)

for _, row in results_df.iterrows():
    gt = row["ground_truth_entities"]
    pred = row["predicted_entities"]
    for ent in gt & pred:
        label_tp[ent[0]] += 1
    for ent in pred - gt:
        label_fp[ent[0]] += 1
    for ent in gt - pred:
        label_fn[ent[0]] += 1

all_labels = sorted(set(label_tp) | set(label_fp) | set(label_fn))
print(f"{'Label':6s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Support':>8s}")
for label in all_labels:
    t, f_p, f_n = label_tp[label], label_fp[label], label_fn[label]
    p = t / (t + f_p) if (t + f_p) > 0 else 0.0
    r = t / (t + f_n) if (t + f_n) > 0 else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    support = t + f_n
    print(f"{label:6s} {p:>10.3f} {r:>10.3f} {f:>10.3f} {support:>8d}")

Label   Precision     Recall         F1  Support
LOC         0.460      0.576      0.511       99
MISC        0.217      0.108      0.144       93
ORG         0.342      0.364      0.352      228
PER         0.811      0.808      0.809     1092


## 9. Row-by-row diff

Shows ground truth vs. predicted entity strings side by side for every
test tweet, with a status flag - useful for spot-checking specific
errors rather than only looking at aggregate numbers. `MISMATCH` rows
are worth reading individually; they're where the model's predictions
and the ground truth disagree.

In [13]:
def row_status(row):
    if row["ground_truth_entities"] == row["predicted_entities"]:
        return "EXACT MATCH" if row["ground_truth_entities"] else "CORRECT (no entities)"
    return "MISMATCH"

results_df["status"] = results_df.apply(row_status, axis=1)
print(results_df["status"].value_counts())
print()

pd.set_option("display.max_colwidth", 80)
results_df[["id", "ground_truth_string", "predicted_string", "status"]].head(20)

status
EXACT MATCH              596
MISMATCH                 497
CORRECT (no entities)    357
Name: count, dtype: int64



,id,ground_truth_string,predicted_string,status
0,2816,PER/Christian Bale;MISC/The Dark Knight Rises;,PER/Christian Bale;,MISMATCH
1,2817,PER/KANYE WEST;,PER/KANYE;,MISMATCH
2,2818,PER/Netanyahu;PER/Ackerman;,PER/Netanyahu;PER/Ackerman;,EXACT MATCH
3,2819,PER/Netanyahu;,PER/Netanyahu;,EXACT MATCH
4,2820,,,CORRECT (no entities)
5,2821,"LOC/Ontario, Canada;",LOC/Ontario;LOC/Canada;,MISMATCH
6,2822,,,CORRECT (no entities)
7,2823,PER/Bill Maher;LOC/Caesars Atlantic;,PER/Bill Maher's;LOC/Caesars Atlantic;,MISMATCH
8,2824,PER/Billy Cox;,PER/Billy Cox;,EXACT MATCH
9,2825,PER/Joel Osteen;,PER/Joel Osteen;,EXACT MATCH


## 10. Inspect mismatches in detail

For each mismatched tweet: the original text, and exactly which
entities were missed (false negatives) vs. incorrectly predicted
(false positives).

## 11. Save full results to CSV

In [ ]:
OUTPUT_PATH = "test_set_predictions_vs_ground_truth.csv"

save_df = results_df.copy()
save_df["ground_truth_entities"] = save_df["ground_truth_entities"].apply(lambda s: "; ".join(f"{l}/{t}" for l, t in s))
save_df["predicted_entities"] = save_df["predicted_entities"].apply(lambda s: "; ".join(f"{l}/{t}" for l, t in s))
save_df[["id", "text", "ground_truth_string", "predicted_string", "status"]].to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(save_df)} rows to {OUTPUT_PATH}")